In [ ]:
%load_ext watermark


In [ ]:
import itertools as it
import os

import matplotlib as mpl
from matplotlib import pyplot as plt
from phyloframe import _auxlib as pfa
from phyloframe import legacy as pfl
from pyfonts import load_google_font
import seaborn as sns
from teeplot import teeplot as tp

import pylib  # noqa: F401


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get("NOTEBOOK_NAME", "2026-03-04-fossil-layers")
teeplot_subdir


In [ ]:
pfa.seed_random(1)


In [ ]:
font = load_google_font("Merriweather", weight=300)
mpl.font_manager.fontManager.addfont(font.get_file())
plt.rcParams["font.family"] = font.get_name()


## Prep Data


In [ ]:
df_pure = pfl.alifestd_join_roots(
    pylib.read_parquet_with_retry("https://osf.io/download/pfvsg"),
)
df_pure


In [ ]:
df_sweep = pfl.alifestd_join_roots(
    pylib.read_parquet_with_retry("https://osf.io/download/nk69s"),
)
df_sweep


In [ ]:
dfs = []
for df in (df_pure, df_sweep):
    df["x"] = df["position"] // df["nCol"]
    df["x_"] = df["x"] / df["nRow"]
    df["y"] = df["position"] % df["nCol"]
    df["y_"] = df["y"] / df["nCol"]

    df["origin_time"] = df["dstream_rank"]

    dfs.append(df)

df_pure, df_sweep = dfs


## Plot Layer Tree


In [ ]:
for s, regime, layout in it.product(
    (1, 3),
    ("pure", "sweep"),
    ("vertical", "horizontal", "radial"),
):
    df = df_pure if regime == "pure" else df_sweep
    with tp.teed(
        pylib.tree.draw_scatter_tree,
        pfl.alifestd_downsample_tips_asexual(df, n_downsample=5_000),
        hue="layer",
        layout=layout,
        scatter_kws=dict(
            alpha=0.8,
            edgecolor="none",
            palette=sns.color_palette("gist_earth", 120)[10:-10],
            s=s,
        ),
        scatter_shuffle=1,
        tree_kws=dict(
            edge=dict(
                color="gainsboro",
                linewidth=0.5,
            ),
            margins=-0.05,
        ),
        teeplot_outattrs={"regime": regime, "s": s},
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        teed.invert_yaxis()
        teed.figure.set_size_inches(3, 1.33)


In [ ]:
for regime, layout in it.product(
    ("pure", "sweep"),
    ("vertical",),
):
    df = df_pure if regime == "pure" else df_sweep
    with tp.teed(
        pylib.tree.draw_scatter_tree,
        pfl.alifestd_downsample_tips_asexual(df, n_downsample=5_000),
        hue="layer",
        layout=layout,
        scatter_kws=dict(
            alpha=0.8,
            edgecolor="none",
            palette=sns.color_palette("gist_earth", 120)[10:-10],
            s=6,
        ),
        scatter_shuffle=1,
        tree_kws=dict(
            edge=dict(
                color="gainsboro",
                linewidth=0.5,
            ),
        ),
        teeplot_outattrs={"regime": regime},
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        pass


## Fill Template with Scatter Tree Plot


In [ ]:
import pymupdf

positions_path = "assets/wse-layer-phylo-positions.pdf"
template_path = "assets/wse-layer-phylo-template.pdf"

positions_doc = pymupdf.open(positions_path)
print(f"Positions: {len(positions_doc)} page(s), size {positions_doc[0].rect}")

template_doc = pymupdf.open(template_path)
print(f"Template:  {len(template_doc)} page(s), size {template_doc[0].rect}")


In [ ]:
target_color = "#DEADBE"


def hex_to_rgb_float(hex_color):
    h = hex_color.lstrip("#")
    return tuple(int(h[i : i + 2], 16) / 255.0 for i in (0, 2, 4))


def find_rects_by_color(page, hex_color, tol=2 / 255):
    target = hex_to_rgb_float(hex_color)
    rects = []
    for path in page.get_drawings():
        fill = path.get("fill")
        if fill is None or len(fill) != 3:
            continue
        if all(abs(fill[i] - target[i]) < tol for i in range(3)):
            rects.append(path["rect"])
    return rects


positions_page = positions_doc[0]
deadbe_rects = find_rects_by_color(positions_page, target_color)
for r in deadbe_rects:
    print(f"  deadbe ({target_color}): {r}")

positions_doc.close()


In [ ]:
scatter_tree_path = os.path.join(
    "teeplots",
    teeplot_subdir,
    "hue=layer+layout=vertical+regime=pure+viz=draw-scatter-tree+ext=.pdf",
)
print(f"Loading scatter tree from: {scatter_tree_path}")
scatter_doc = pymupdf.open(scatter_tree_path)

page = template_doc[0]
for rect in deadbe_rects:
    page.show_pdf_page(rect, scatter_doc, 0, rotate=180)
    print(f"  Inserted rotated scatter tree at {rect}")

output_destination = f"teeplots/{teeplot_subdir}/"
os.makedirs(output_destination, exist_ok=True)
output_path = os.path.join(output_destination, "wse-layer-phylo-filled.pdf")
template_doc.save(output_path, garbage=4, deflate=True)
template_doc.close()
scatter_doc.close()
print(f"\nSaved to {output_path}")


In [ ]:
filled_doc = pymupdf.open(output_path)
dpi = 600
mat = pymupdf.Matrix(dpi / 72, dpi / 72)
pix = filled_doc[0].get_pixmap(matrix=mat, alpha=False)
png_path = output_path.replace(".pdf", ".png")
pix.save(png_path)
filled_doc.close()
print(f"Saved {pix.width}x{pix.height} @ {dpi} DPI to {png_path}")
